# Atividade: construindo um classificador de frutas

Você receberá exemplos de maçãs, tangerinas, laranjas e limões. Sua tarefa é criar, por conta própria, uma função que tente reconhecer cada fruta usando `if`, `elif` e `else`.

Não existe um roteiro indicando quais características ou intervalos usar. Parte da atividade é descobrir uma estratégia, perceber os casos difíceis e decidir como lidar com eles. Execute as células na ordem e não procure o gabarito do teste antes de congelar suas regras. Ao longo do notebook, usaremos nomes comuns em aprendizado de máquina — como `train`, `test`, `X`, `y`, `fit` e `predict` — e construiremos o significado de cada um.

## 1. Conhecer os exemplos disponíveis

O código abaixo lê um arquivo CSV com Pandas. Cada linha representa uma fruta. As colunas `mass`, `width`, `height` e `color_score` descrevem a fruta; `fruit_name` informa seu nome. Nos dados originais, tangerina aparece como `mandarin`.

In [1]:
import pandas as pd

TRAIN_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/train.csv'
train = pd.read_csv(TRAIN_URL)

FEATURES = ['mass', 'width', 'height', 'color_score']
TARGET = 'fruit_name'

X_train = train[FEATURES]
y_train = train[TARGET]

pd.set_option('display.max_rows', None)
display(train)

,id,mass,width,height,color_score,fruit_name
0,F001,192,8.4,7.3,0.55,apple
1,F002,180,8.0,6.8,0.59,apple
2,F003,86,6.2,4.7,0.80,mandarin
3,F004,84,6.0,4.6,0.79,mandarin
4,F005,80,5.8,4.3,0.77,mandarin
5,F006,80,5.9,4.3,0.81,mandarin
6,F007,178,7.1,7.8,0.92,apple
7,F008,172,7.4,7.0,0.89,apple
8,F009,172,7.1,7.6,0.92,apple
9,F010,154,7.0,7.1,0.88,apple


In [39]:
# Examine a tabela completa com calma.
# Esta célula fica livre para qualquer exploração que você decidir fazer.

tangerinas = train[train['fruit_name'] == 'mandarin']
media_mass_tangerin = tangerinas['mass'].mean()
print(f"A média do mass das tangerinas é: {media_mass_tangerin:.4f}")
media_width_tangerin = tangerinas['width'].mean()
print(f"A média do width das tangerinas é: {media_width_tangerin:.4f}")
media_height_mandarin = tangerinas['height'].mean()
print(f"A média do height das tangerinas é: {media_height_mandarin:.4f}")
media_color_mandarin = tangerinas['color_score'].mean()
print(f"A média do color_score das tangerinas é: {media_color_mandarin:.4f}")
area_tangerinas = tangerinas['mass'].mean() * tangerinas['width'].mean()
print(f"area média das tangerinas: "+ str(area_tangerinas))

print("------------------------")

macas = train[train['fruit_name'] == 'apple']
media_mass_apple = macas['mass'].mean()
print(f"A média do mass das maçãs é: {media_mass_apple:.4f}")
media_width_apples = macas['width'].mean()
print(f"A média do widht das maçãs é: {media_width_apples:.4f}")
media_height_apple = macas['height'].mean()
print(f"A média do height das maçãs é: {media_height_apple:.4f}")
media_color_apple = macas['color_score'].mean()
print(f"A média do color_score das maçãs é: {media_color_apple:.4f}")
area_macas = macas['mass'].mean() * macas['width'].mean()
print(f"area média das macas: "+ str(area_macas))

print("------------------------")

laranjas = train[train['fruit_name'] == 'orange']
media_mass_orange = laranjas['mass'].mean()
print(f"A média do mass das laranjas é: {media_mass_orange:.4f}")
media_width_orange = laranjas['width'].mean()
print(f"A média do width das laranjas é: {media_width_orange:.4f}")
media_height_orange = laranjas['height'].mean()
print(f"A média do height das laranjas é: {media_height_orange:.4f}")
media_color_orange = laranjas['color_score'].mean()
print(f"A média do color_score das laranjas é: {media_color_orange:.4f}")
area_laranjas = laranjas['mass'].mean() * laranjas['width'].mean()
print(f"area média das laranjas: "+ str(area_laranjas))

print("------------------------")

limoes = train[train['fruit_name'] == 'lemon']
media_mass_lemon = limoes['mass'].mean()
print(f"A média do mass dos limões é: {media_mass_lemon:.4f}")
media_width_lemon = limoes['width'].mean()
print(f"A média do width dos limões é: {media_color:.4f}")
media_height_lemon = limoes['height'].mean()
print(f"A média do height dos limões é: {media_height_lemon:.4f}")
media_color_lemon = limoes['color_score'].mean()
print(f"A média do color_score dos limões é: {media_color_lemon:.4f}")
area_limoes = limoes['mass'].mean() * limoes['width'].mean()
print(f"area média das limoes: "+ str(area_limoes))


A média do mass das tangerinas é: 82.5000
A média do width das tangerinas é: 5.9750
A média do height das tangerinas é: 4.4750
A média do color_score das tangerinas é: 0.7925
area média das tangerinas: 492.93749999999994
------------------------
A média do mass das maçãs é: 163.7333
A média do widht das maçãs é: 7.4867
A média do height das maçãs é: 7.3000
A média do color_score das maçãs é: 0.7833
area média das macas: 1225.8168888888888
------------------------
A média do mass das laranjas é: 187.8750
A média do width das laranjas é: 7.4812
A média do height das laranjas é: 7.9188
A média do color_score das laranjas é: 0.7725
area média das laranjas: 1405.5398437499998
------------------------
A média do mass dos limões é: 149.5385
A média do width dos limões é: 0.7192
A média do height dos limões é: 8.8077
A média do color_score dos limões é: 0.7192
area média das limoes: 980.0520710059172


Examine todas as linhas antes de programar. Procure relações entre os valores e o nome da fruta, experimente hipóteses e anote o que considerar importante. O notebook não calculará intervalos nem indicará quais características usar: decidir isso faz parte do problema.

In [42]:
# Para cada fruta, exibe o menor (min) e o maior (max) valor de todos os atributos
limites_frutas = train.groupby('fruit_name')[['mass', 'width', 'height', 'color_score']].agg(['min', 'max'])
display(limites_frutas)

mass      width      height       color_score      
            min  max   min  max    min   max         min   max
fruit_name                                                    
apple       140  192   7.0  8.4    6.8   7.8        0.55  0.92
lemon       116  216   5.9  7.3    7.5  10.5        0.70  0.74
mandarin     80   86   5.8  6.2    4.3   4.7        0.77  0.81
orange      140  356   6.7  9.2    7.1   9.4        0.72  0.82

Quando tiver uma primeira ideia, registre-a com suas palavras. Não é necessário encontrar uma separação perfeita antes de começar a programar.

**Minha estratégia inicial:** ...

## 2. Escrever as regras

Em aprendizado de máquina, costumamos chamar as entradas de `X` e as respostas conhecidas de `y`. Por isso, separamos acima `X_train` e `y_train`: são as entradas e respostas disponíveis durante o ajuste.

Mais adiante, uma biblioteca poderá aprender automaticamente com uma chamada como `model.fit(X_train, y_train)` e gerar respostas com `model.predict(X_test)`. Nesta atividade, você fará o papel do `fit` manualmente: observará os exemplos e decidirá as condições da função `predict_fruit`.

Complete a função abaixo. Ela deve devolver exatamente um destes nomes: `apple`, `mandarin`, `orange` ou `lemon`. Você pode criar quantas condições forem necessárias, mas deve conseguir explicar cada uma delas.

In [55]:
def predict_fruit(mass, width, height, color_score):
    # Regra para tangerina (mandarin)
    if (80 <= mass <= 86) and (5.8 <= width <= 6.2) and (4.3 <= height <= 4.7) and (0.77 <= color_score <= 0.81):
        return 'mandarin'

    # Regra para limão (lemon)
    elif (116 <= mass <= 216) and (5.9 <= width <= 7.3) and (7.5 <= height <= 10.5) and (0.70 <= color_score <= 0.74):
        return 'lemon'

    # Regra para laranja (orange)
    elif (140 <= mass <= 356) and (6.7 <= width <= 9.2) and (7.1 <= height <= 9.4) and (0.72 <= color_score <= 0.82):
        return 'orange'

    # Regra para maçã (apple)
    elif (140 <= mass <= 192) and (7.0 <= width <= 8.4) and (6.8 <= height <= 7.8) and (0.55 <= color_score <= 0.92):
        return 'apple'

    else:
        # Caso o exemplo não caia em nenhuma regra exata (um valor fora do treino)
        return 'erro'

# Teste manualmente uma fruta antes de aplicar a função a todas as linhas.
print(predict_fruit(mass=192, width=8.4, height=7.3, color_score=0.55))

apple


In [56]:
def predict(X):
    return X.apply(
        lambda row: predict_fruit(
            row['mass'],
            row['width'],
            row['height'],
            row['color_score'],
        ),
        axis=1,
    )

train_results = train[['id'] + FEATURES].copy()
train_results['y_true'] = y_train
train_results['y_pred'] = predict(X_train)
train_results['correct'] = train_results['y_pred'] == train_results['y_true']
train_results.head()

,id,mass,width,height,color_score,y_true,y_pred,correct
0,F001,192,8.4,7.3,0.55,apple,apple,True
1,F002,180,8.0,6.8,0.59,apple,apple,True
2,F003,86,6.2,4.7,0.80,mandarin,mandarin,True
3,F004,84,6.0,4.6,0.79,mandarin,mandarin,True
4,F005,80,5.8,4.3,0.77,mandarin,mandarin,True


In [58]:
train_accuracy = train_results['correct'].mean()
print(f'Train accuracy: {train_accuracy:.1%}')
display(train_results.loc[~train_results['correct']])

Train accuracy: 97.9%


,id,mass,width,height,color_score,y_true,y_pred,correct
10,F011,164,7.3,7.7,0.7,apple,lemon,False


Agora você pode revisar livremente suas regras olhando apenas para `train`. Cada tentativa pode corrigir alguns casos e criar novos erros. Continue até chegar a uma solução que considere razoável e que consiga explicar. Depois registre:

- qual resultado você conseguiu em `train`;
- quais casos continuaram difíceis;
- por que decidiu encerrar os ajustes nessa versão.

## 3. Congelar as regras

Antes de abrir os exemplos novos:

1. anote sua acurácia nos exemplos conhecidos;
2. estime quantas das 11 frutas novas sua função acertará;
3. registre uma justificativa curta;
4. salve esta versão do notebook e não altere mais a função até avaliar o teste.

**Minha estimativa:** ... acertos em 11.

**Minha justificativa:** ...

In [59]:
#Train accuracy: 97.9%
#estimativa para 11 frutas=0.979x11=10,76 -> chuto que 10

## 4. Fazer previsões para frutas novas

O próximo arquivo não contém `fruit_name`. Gere as previsões sem alterar as regras.

In [60]:
TEST_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/test.csv'
test = pd.read_csv(TEST_URL)
X_test = test[FEATURES]
test

,id,mass,width,height,color_score
0,T01,176,7.4,7.2,0.60
1,T02,76,5.8,4.0,0.81
2,T03,166,6.9,7.3,0.93
3,T04,168,7.5,7.6,0.73
4,T05,170,7.6,7.9,0.88
5,T06,362,9.6,9.2,0.74
6,T07,164,7.2,7.0,0.80
7,T08,150,7.1,7.9,0.75
8,T09,194,7.2,10.3,0.70
9,T10,132,5.8,8.7,0.73


In [61]:
test_predictions = test[['id']].copy()
test_predictions['prediction'] = predict(X_test)
test_predictions.to_csv('fruit_predictions.csv', index=False)
display(test_predictions)

try:
    from google.colab import files
    files.download('fruit_predictions.csv')
except ImportError:
    print('The file fruit_predictions.csv was saved in the current folder.')

,id,prediction
0,T01,apple
1,T02,erro
2,T03,erro
3,T04,orange
4,T05,erro
5,T06,erro
6,T07,apple
7,T08,orange
8,T09,lemon
9,T10,erro


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. O que você acabou de fazer?

Agora podemos dar nome às partes da experiência:

- massa, largura, altura e medida de cor são as **características** (*features*);
- o nome da fruta é a **classe** ou **rótulo** (*label*);
- `X` reúne as **características** (*features*) usadas como entrada;
- `y` reúne a **classe** ou **rótulo** (*label*) que queremos prever;
- sua função de regras é um **modelo**: `predict(X)` transforma características em previsões;
- `X_train` e `y_train` formam os **dados de treino**;
- você fez o papel de `fit` manualmente, ajustando o modelo com esses dados;
- `X_test` contém os **dados de teste**, que o modelo ainda não tinha visto;
- funcionar em exemplos novos é o que chamamos de **generalização**.

Você ainda não treinou automaticamente um algoritmo de aprendizado de máquina. A atividade separou as etapas para que fosse possível sentir a dificuldade de formular regras e avaliar se elas generalizam.

## 6. Avaliar depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de entregar ou congelar suas previsões.

In [64]:
ANSWER_KEY_URL = ''  # Cole aqui o link direto liberado pelo professor.

if not ANSWER_KEY_URL.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    answer_key = pd.read_csv(ANSWER_KEY_URL)
    test_results = test_predictions.merge(answer_key, on='id', validate='one_to_one')
    test_results = test_results.rename(
        columns={'fruit_name': 'y_true', 'prediction': 'y_pred'}
    )
    test_results['correct'] = test_results['y_pred'] == test_results['y_true']
    test_accuracy = test_results['correct'].mean()

    print(f'Test accuracy: {test_accuracy:.1%}')
    print('\nAcurácia por classe:')
    accuracy_by_class = test_results.groupby('y_true')['correct'].mean()
    display(accuracy_by_class.rename('accuracy'))

    print('Matriz de confusão:')
    confusion_matrix = pd.crosstab(
        test_results['y_true'],
        test_results['y_pred'],
        rownames=['y_true'],
        colnames=['y_pred'],
        dropna=False,
    )
    display(confusion_matrix)

    print('Erros:')
    display(test_results.loc[~test_results['correct']])

O gabarito ainda não foi informado. Preserve suas previsões e volte depois.


## 7. Fechamento

Responda com suas palavras:

1. O resultado do teste ficou acima ou abaixo de sua estimativa?
2. Compare a acurácia nos dados de treino com a acurácia no teste. O que mudou?
3. Qual classe teve o pior resultado? A acurácia total deixava isso evidente?
4. Observe os erros. Há frutas com valores fora dos intervalos vistos no treino?
5. O que você mudaria nas regras? Por que não devemos usar o mesmo teste repetidamente para fazer esses ajustes?
6. Que vantagem um algoritmo de treinamento poderia ter sobre o ajuste manual? Que cuidado ainda continuaria necessário?